# Notebook 02 — FTC Case Acquisition

**Purpose** Scrapes the FTC Legal Library (ftc.gov cases & proceedings, "Privacy and Security" topic filter 1420) into the raw US case table. First link in the US chain: nb02 → nb03 → nb04 → nb08 → nb09.

**Inputs**
- ftc.gov listing + case detail pages (live scrape; ftc.gov 403s headless browsers, so this uses Playwright's HTTP request context with a browser User-Agent, not a headless page)

**Outputs**
- `data/ftc_cases_raw.csv` (339 rows, pre-cleaning)
- `data/ftc_cases_clean.csv` (339 rows × 11 cols — dates, matter numbers, topic tags, keyword-extracted statutes, naive penalty extraction)

**Notes for rerunning**
- Listing mixes case and public-statement nodes; only `node--type-case` rows are kept
- The 339 here becomes 337 in the final corpus (2 dropped downstream as non-case records)
- `statutes` / `penalty_usd` extracted here are FIRST-PASS only: statutes are keyword regex over summary+tags, penalty is "largest dollar amount in summary." Both are superseded by nb03 (press-release extraction) and nb04/nb07 (label finalization) — do not trust these columns directly
- Live scrape: results will drift as FTC adds cases; corpus snapshot is July 2026

In [1]:
##Imports

from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd
import asyncio
import re
import os

In [2]:
##Scrapes FTC cases & proceedings filtered to Privacy and Security (topic 1420)
##ftc.gov 403s headless browsers, so this uses Playwright's HTTP request context instead
##results mix case and public-statement nodes; only node--type-case rows are kept

BASE = "https://www.ftc.gov"
LIST_URL = BASE + "/legal-library/browse/cases-proceedings?search=&field_consumer_protection_topics=1420&items_per_page=100&page={}"
UA = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"

def parse_listing(html):
    soup = BeautifulSoup(html, "html.parser")
    results = soup.select("article.node--view-mode-search-result")
    cases = []
    for art in results:
        if "node--type-case" not in art.get("class", []):
            continue
        def field(name):
            el = art.select_one(f".field--name-{name} .field__item")
            return el.get_text(" ", strip=True) if el else None
        title = art.select_one("h3.node-title a")
        time_el = art.select_one(".field--name-field-date time")
        cases.append({
            "case_name": title.get_text(strip=True) if title else None,
            "url": BASE + title["href"] if title else None,
            "date": time_el["datetime"] if time_el else None,
            "case_type": field("field-case-action-type"),
            "matter_number": field("field-matter-number"),
            "case_status": field("field-case-status"),
            "summary": field("body"),
        })
    return cases, len(results)

async def scrape_listings():
    async with async_playwright() as p:
        req = await p.request.new_context(extra_http_headers={"User-Agent": UA})
        cases, page_num = [], 0
        while True:
            resp = await req.get(LIST_URL.format(page_num))
            batch, n_results = parse_listing(await resp.text())
            if n_results == 0:
                break
            cases.extend(batch)
            print(f"page {page_num}: {len(batch)} cases of {n_results} results (total {len(cases)})")
            page_num += 1
            await asyncio.sleep(1)
        await req.dispose()
    return pd.DataFrame(cases)

df = await scrape_listings()
print(df.shape)

page 0: 57 cases of 100 results (total 57)


page 1: 50 cases of 100 results (total 107)


page 2: 66 cases of 100 results (total 173)


page 3: 76 cases of 100 results (total 249)


page 4: 75 cases of 100 results (total 324)


page 5: 15 cases of 29 results (total 339)


(339, 7)


In [3]:
##Fetches each case detail page for topic tags and full case title

async def fetch_details(urls, concurrency=5):
    results = {}
    async with async_playwright() as p:
        request = await p.request.new_context(extra_http_headers={"User-Agent": UA})
        sem = asyncio.Semaphore(concurrency)

        async def fetch(url):
            async with sem:
                try:
                    resp = await request.get(url, timeout=30000)
                    if resp.status != 200:
                        raise Exception(f"status {resp.status}")
                    html = await resp.text()
                except Exception:
                    results[url] = {"tags": None, "long_title": None}
                    return
                soup = BeautifulSoup(html, "html.parser")
                tags = [a.get_text(strip=True) for a in soup.select(".view-tags .views-field-name a")]
                lt = soup.select_one(".field--name-field-long-title .field__item")
                results[url] = {
                    "tags": "; ".join(tags) if tags else None,
                    "long_title": lt.get_text(" ", strip=True) if lt else None,
                }

        await asyncio.gather(*[fetch(u) for u in urls])
        await request.dispose()
    return results

details = await fetch_details(df["url"].tolist())
df["tags"] = df["url"].map(lambda u: details[u]["tags"])
df["long_title"] = df["url"].map(lambda u: details[u]["long_title"])
print(f"detail pages fetched: {df['tags'].notna().sum()} of {len(df)} with tags")

detail pages fetched: 339 of 339 with tags


In [4]:
##Cleans; extracts statutes cited and penalty amounts from summary text

df_raw = df.copy()

for col in ["case_name", "long_title", "summary", "tags", "case_type", "case_status"]:
    df[col] = df[col].str.replace(r"\s+", " ", regex=True).str.strip()

df["date"] = pd.to_datetime(df["date"], utc=True).dt.date
df["matter_number"] = df["matter_number"].str.replace(" ", "", regex=False)

# statute keywords matched against summary + tags + case name
STATUTE_PATTERNS = {
    "FTC Act Section 5": r"section 5|ftc act|unfair or deceptive|deceptive practices",
    "COPPA": r"coppa|children'?s online privacy",
    "GLBA": r"gramm[- ]leach[- ]bliley|glba|safeguards rule|financial privacy rule",
    "FCRA": r"fair credit reporting|fcra|credit reporting",
    "Health Breach Notification Rule": r"health breach notification",
    "Telemarketing Sales Rule": r"telemarketing sales rule|do not call",
    "ROSCA": r"rosca|restore online shoppers",
    "CAN-SPAM": r"can[- ]spam",
    "Privacy Shield": r"privacy shield",
    "Safe Harbor": r"safe harbor",
    "Red Flags Rule": r"red flags rule",
}

def extract_statutes(row):
    text = " ".join(v for v in [row["summary"], row["tags"], row["case_name"]] if isinstance(v, str)).lower()
    found = [name for name, pat in STATUTE_PATTERNS.items() if re.search(pat, text)]
    return "; ".join(found) if found else None

df["statutes"] = df.apply(extract_statutes, axis=1)

# largest dollar amount mentioned in the summary, as USD
def extract_penalty(text):
    if not isinstance(text, str):
        return None
    amounts = []
    for m in re.finditer(r"\$(\d[\d,]*(?:\.\d+)?)\s*(billion|million|thousand)?", text, re.I):
        val = float(m.group(1).replace(",", ""))
        mult = {"billion": 1e9, "million": 1e6, "thousand": 1e3}.get((m.group(2) or "").lower(), 1)
        amounts.append(val * mult)
    return max(amounts) if amounts else None

df["penalty_usd"] = df["summary"].map(extract_penalty)
print(df[["statutes", "penalty_usd"]].notna().sum())

statutes       201
penalty_usd     25
dtype: int64


In [5]:
##Saves raw and clean to .csv

DATA_DIR = "/Users/nic/Documents/MM2/data"
os.makedirs(DATA_DIR, exist_ok=True)
df_raw.to_csv(f"{DATA_DIR}/ftc_cases_raw.csv", index=False)
df.to_csv(f"{DATA_DIR}/ftc_cases_clean.csv", index=False)
print(df.shape)
df.head()

(339, 11)


,case_name,url,date,case_type,matter_number,case_status,summary,tags,long_title,statutes,penalty_usd
0,"Amazon.com, Inc., U.S. v.",https://www.ftc.gov/legal-library/browse/cases...,2026-06-30,Federal,2523024,Pending,Amazon will pay $2.25 million in civil penalti...,Consumer Protection; Bureau of Consumer Protec...,"UNITED STATES OF AMERICA, Plaintiff v. AMAZON....",FCRA,2250000.0
1,"FTC v Kochava, Inc.",https://www.ftc.gov/legal-library/browse/cases...,2026-06-26,Federal,NaN,Pending,The FTC will prohibit data broker Kochava and ...,Consumer Protection; Bureau of Consumer Protec...,"Federal Trade Commission, Plaintiff, V. Kochav...",NaN,NaN
2,"Illuminate Education, Inc., In the Matter of",https://www.ftc.gov/legal-library/browse/cases...,2026-06-05,Administrative,2223105,Under Order,The Federal Trade Commission will require educ...,Consumer Protection; Bureau of Consumer Protec...,"In the Matter of ILLUMINATE EDUCATION, INC., a...",NaN,NaN
3,"Twitter, Inc., a corporation",https://www.ftc.gov/legal-library/browse/cases...,2026-06-03,Administrative,0923093,NaN,NaN,Consumer Protection; Bureau of Consumer Protec...,"In the Matter of Twitter, Inc.,a corporation",NaN,NaN
4,"CMG Media Corporation, In the Matter of",https://www.ftc.gov/legal-library/browse/cases...,2026-05-21,Administrative,2423029,Pending,"The FTC will require Cox Media Group, MindSift...",Consumer Protection; Bureau of Consumer Protec...,In the Matter of CMG Media Corporation,NaN,930000.0
